# GraphMask across historical periods: relation-group comparisons

This notebook compares **independently trained two-layer R-GATs** across induced historical life-period graphs. It covers only the binary and multi-group relation vocabularies; exact relations are intentionally deferred.

The main figure uses the global hard-retained-message share: $K_{group}/K_{all}$ within one model and one GraphMask layer. Thus each completed period/model panel sums to 100%. It is a frozen-model dependence measure, not a causal effect of a social relationship.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

MPL_CACHE = Path('/tmp/rgcn_matplotlib')
MPL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPL_CACHE))

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 50)


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'runs_graphmask').exists() and (candidate / 'visualization').exists():
            return candidate
    raise FileNotFoundError('Could not find the project root.')


PROJECT_ROOT = find_project_root(Path.cwd())
REPORT_ROOT = PROJECT_ROOT / 'runs_graphmask' / 'level1' / 'life_period_relation_representations_rgat_v1'
VISUALIZATION_DIR = PROJECT_ROOT / 'visualization'
VISUALIZATION_DIR.mkdir(parents=True, exist_ok=True)

# Valid alternatives: retained_edge_share (main figure) or hard_retention_rate (auxiliary).
METRIC = os.environ.get('GRAPHMASK_PERIOD_GROUP_METRIC', 'retained_edge_share')
METRIC_META = {
    'retained_edge_share': {
        'label': 'Global hard-retained message share',
        'filename': 'global_retained_share',
        'caption': ('Cell = K_group / K_all for this model and Layer. Each completed period/model panel sums to 100%.'),
        'normalise_each_row': True,
    },
    'hard_retention_rate': {
        'label': 'Hard retention rate',
        'filename': 'hard_retention_rate',
        'caption': ('Cell = K_group / N_group. This is a within-group retention rate and does not sum to 100%.'),
        'normalise_each_row': False,
    },
}
if METRIC not in METRIC_META:
    raise ValueError(f'Unknown METRIC={METRIC!r}; choose one of {sorted(METRIC_META)}.')

PERIODS = (
    ('ancient_history', 'Ancient\n(≤500)'),
    ('post_classical_history', 'Post-Classical\n(501–1500)'),
    ('mid_modern_period', 'Mid-Modern\n(1501–1800)'),
    ('contemporary_period', 'Contemporary\n(≥1801)'),
)
PERIOD_IDS = [identifier for identifier, _ in PERIODS]
PERIOD_LABELS = {identifier: label for identifier, label in PERIODS}

GROUPS = {
    'binary': ('inherited_ties', 'acquired_ties'),
    'multi_group': (
        'inherited', 'intimate_partnership', 'education_mentorship',
        'professional_collaboration', 'influence_succession',
        'religious_ordination', 'other_acquired',
    ),
}
GROUP_LABELS = {
    'inherited_ties': 'inherited',
    'acquired_ties': 'acquired',
    'inherited': 'inherited',
    'intimate_partnership': 'intimate\npartnership',
    'education_mentorship': 'education /\nmentorship',
    'professional_collaboration': 'professional\ncollaboration',
    'influence_succession': 'influence /\nsuccession',
    'religious_ordination': 'religious\nordination',
    'other_acquired': 'other\nacquired',
}

print(f'Project root: {PROJECT_ROOT}')
print(f'Report root: {REPORT_ROOT}')
print(f'Metric: {METRIC}')

In [ ]:
REQUIRED_REPORT_FILES = {'relations_base.csv', 'relations_directed.csv', 'test_metrics.json', 'manifest.json'}


def parse_job_name(path: Path) -> tuple[str, str]:
    name = path.parents[2].name
    prefix = 'rgat__period_'
    if not name.startswith(prefix):
        raise ValueError(f'Unexpected GraphMask job directory: {name}')
    return name.removeprefix(prefix).rsplit('__', 1)


records = []
audit_rows = []
for report_dir in sorted(REPORT_ROOT.glob('rgat__period_*__*/seed_42/test_report')):
    missing = [filename for filename in REQUIRED_REPORT_FILES if not (report_dir / filename).is_file()]
    if missing:
        raise ValueError(f'Incomplete report {report_dir}: missing {missing}')
    period, representation = parse_job_name(report_dir / 'relations_base.csv')
    table = pd.read_csv(report_dir / 'relations_base.csv')
    metrics = json.loads((report_dir / 'test_metrics.json').read_text(encoding='utf-8'))
    validation_path = report_dir.parent / 'validation.json'
    validation = json.loads(validation_path.read_text(encoding='utf-8')) if validation_path.is_file() else {}
    table['period'] = period
    table['representation'] = representation
    records.append(table)
    original_f1 = float(metrics['original']['macro_f1'])
    masked_f1 = float(metrics['masked']['macro_f1'])
    audit_rows.append({
        'period': period,
        'representation': representation,
        'test_nodes': int(metrics['roots']),
        'test_macro_f1_original': original_f1,
        'test_macro_f1_masked': masked_f1,
        'test_relative_macro_f1_difference': abs(original_f1 - masked_f1) / max(abs(original_f1), 1e-12),
        'test_mean_kl': float(metrics['mean_kl']),
        'validation_relative_macro_f1_difference': validation.get('relative_macro_f1_difference'),
    })

if not records:
    raise FileNotFoundError(f'No complete GraphMask reports found under {REPORT_ROOT}')

relations = pd.concat(records, ignore_index=True)
audit = pd.DataFrame(audit_rows).sort_values(['period', 'representation'])

# A completed binary Ancient probe does not exist. Preserve the absence as NaN in plots.
expected_jobs = pd.MultiIndex.from_product([PERIOD_IDS, tuple(GROUPS)], names=['period', 'representation'])
completed_jobs = pd.MultiIndex.from_frame(audit[['period', 'representation']])
missing_jobs = expected_jobs.difference(completed_jobs)
assert set(missing_jobs.tolist()) == {('ancient_history', 'binary')}

for (period, representation, layer), frame in relations.groupby(['period', 'representation', 'layer']):
    if METRIC_META[METRIC]['normalise_each_row']:
        total = float(frame[METRIC].sum())
        if not np.isclose(total, 1.0, atol=1e-8):
            raise ValueError(f'{period}/{representation}/Layer {layer}: {METRIC} sums to {total}, not one.')

display(
    audit.assign(
        test_relative_macro_f1_difference=lambda x: x['test_relative_macro_f1_difference'].map(lambda v: f'{v:.2%}'),
        validation_relative_macro_f1_difference=lambda x: x['validation_relative_macro_f1_difference'].map(lambda v: f'{v:.2%}'),
        test_mean_kl=lambda x: x['test_mean_kl'].map(lambda v: f'{v:.4f}'),
    )
)

In [ ]:
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 17,
    'figure.dpi': 160,
    'savefig.dpi': 220,
})


def group_matrix(representation: str, layer: int) -> pd.DataFrame:
    groups = list(GROUPS[representation])
    selected = relations.loc[
        relations['representation'].eq(representation)
        & relations['layer'].eq(layer)
        & relations['relation'].isin(groups),
        ['period', 'relation', METRIC],
    ].copy()
    matrix = selected.pivot(index='period', columns='relation', values=METRIC)
    return matrix.reindex(index=PERIOD_IDS, columns=groups)


def draw_heatmap(ax, matrix: pd.DataFrame, title: str, norm: Normalize) -> object:
    values = matrix.to_numpy(dtype=float)
    image = ax.imshow(values, cmap='Blues', norm=norm, aspect='auto')
    ax.set_title(title, pad=18)
    ax.set_xticks(range(len(matrix.columns)), [GROUP_LABELS[column] for column in matrix.columns], fontsize=11)
    ax.xaxis.tick_top()
    ax.tick_params(axis='x', length=0, pad=9)
    ax.set_yticks(range(len(matrix.index)), [PERIOD_LABELS[index] for index in matrix.index], fontsize=11)
    ax.tick_params(axis='y', length=0, pad=8)

    for row in range(values.shape[0]):
        for col in range(values.shape[1]):
            value = values[row, col]
            if np.isnan(value):
                ax.add_patch(plt.Rectangle((col - 0.5, row - 0.5), 1, 1, facecolor='#f0f0f0', edgecolor='white', hatch='///'))
                ax.text(col, row, 'N/A', ha='center', va='center', color='#555555', fontsize=11)
                continue
            text_colour = 'white' if norm(value) > 0.58 else '#202020'
            ax.text(col, row, f'{value:.1%}', ha='center', va='center', color=text_colour, fontsize=11)

    ax.set_xticks(np.arange(-0.5, len(matrix.columns), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(matrix.index), 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=1.25)
    ax.tick_params(which='minor', bottom=False, left=False)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
    return image


def plot_layer(layer: int) -> Path:
    binary = group_matrix('binary', layer)
    multi_group = group_matrix('multi_group', layer)
    finite = np.concatenate([binary.to_numpy(dtype=float)[~np.isnan(binary.to_numpy(dtype=float))], multi_group.to_numpy(dtype=float)[~np.isnan(multi_group.to_numpy(dtype=float))]])
    vmax = max(float(finite.max()), 1e-12)
    norm = Normalize(vmin=0.0, vmax=vmax)

    fig = plt.figure(figsize=(16, 6.7), constrained_layout=False)
    grid = fig.add_gridspec(1, 3, width_ratios=(2.2, 7.2, 0.28), left=0.08, right=0.93, top=0.78, bottom=0.22, wspace=0.08)
    binary_ax = fig.add_subplot(grid[0, 0])
    multi_ax = fig.add_subplot(grid[0, 1])
    color_ax = fig.add_subplot(grid[0, 2])
    image = draw_heatmap(binary_ax, binary, 'Binary relation vocabulary', norm)
    draw_heatmap(multi_ax, multi_group, 'Multi-group relation vocabulary', norm)
    multi_ax.tick_params(axis='y', which='both', left=False, labelleft=False)
    binary_ax.set_ylabel('Historical life period', fontsize=14)

    colorbar = fig.colorbar(image, cax=color_ax)
    colorbar.ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    colorbar.set_label(f"{METRIC_META[METRIC]['label']} (%)", fontsize=13, labelpad=12)

    fig.suptitle(f'R-GAT GraphMask: relation-group comparison across historical periods — Layer {layer}', fontsize=22, y=0.97)
    fig.text(0.5, 0.075, METRIC_META[METRIC]['caption'] + ' Grey N/A: Ancient binary has no fidelity-qualified GraphMask probe. Ancient multi-group passed validation selection but has a 6.85% test Macro-F1 difference.', ha='center', va='center', fontsize=10.8, wrap=True)

    output = VISUALIZATION_DIR / f'graphmask_period_relation_groups_layer{layer}_{METRIC_META[METRIC]["filename"]}.png'
    fig.savefig(output, bbox_inches='tight', facecolor='white')
    plt.show()
    return output


In [ ]:
output_paths = [plot_layer(layer) for layer in (0, 1)]
for output_path in output_paths:
    print(f'Saved: {output_path}')

### Interpretation boundaries

- A cell is a share of retained **messages**, not a share of unique social ties and not a causal effect.
- Layer 0 and Layer 1 are separate message observations and must not be summed into a unique-edge count.
- The binary and multi-group panels are independently trained models. Their shares can be compared descriptively, but relation collapsing can deduplicate messages, so they are not a partition of one common edge budget.
- The Ancient binary probe failed the pre-specified validation fidelity criterion and is deliberately shown as unavailable rather than zero.